# 02 — Baseline Model: TF-IDF + Logistic Regression

Build a fast, interpretable baseline for fake news detection.

**Target:** 93-95% accuracy

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

PROCESSED_DIR = Path('../data/processed')
MODEL_DIR = Path('../backend/ml/models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load data splits
train_df = pd.read_csv(PROCESSED_DIR / 'train.csv')
val_df = pd.read_csv(PROCESSED_DIR / 'val.csv')
test_df = pd.read_csv(PROCESSED_DIR / 'test.csv')

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

X_train, y_train = train_df['text'], train_df['label_encoded']
X_val, y_val = val_df['text'], val_df['label_encoded']
X_test, y_test = test_df['text'], test_df['label_encoded']

## Build Pipeline

In [ ]:
# TF-IDF + Logistic Regression pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight='balanced',
        random_state=42,
        solver='lbfgs'
    ))
])

print('Training baseline model...')
pipeline.fit(X_train, y_train)
print('Training complete!')

## Evaluate

In [ ]:
# Validation set
y_val_pred = pipeline.predict(X_val)
print('=== Validation Results ===')
print(f'Accuracy:  {accuracy_score(y_val, y_val_pred):.4f}')
print(f'Precision: {precision_score(y_val, y_val_pred):.4f}')
print(f'Recall:    {recall_score(y_val, y_val_pred):.4f}')
print(f'F1 Score:  {f1_score(y_val, y_val_pred):.4f}')
print()
print(classification_report(y_val, y_val_pred, target_names=['Real', 'Fake']))

In [ ]:
# Test set
y_test_pred = pipeline.predict(X_test)
print('=== Test Results ===')
print(f'Accuracy:  {accuracy_score(y_test, y_test_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_test_pred):.4f}')
print(f'Recall:    {recall_score(y_test, y_test_pred):.4f}')
print(f'F1 Score:  {f1_score(y_test, y_test_pred):.4f}')
print()
print(classification_report(y_test, y_test_pred, target_names=['Real', 'Fake']))

In [ ]:
# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_true, y_pred, title in [
    (axes[0], y_val, y_val_pred, 'Validation'),
    (axes[1], y_test, y_test_pred, 'Test')
]:
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Real', 'Fake'])
    disp.plot(ax=ax, cmap='Blues')
    ax.set_title(f'{title} — Confusion Matrix')

plt.tight_layout()
plt.show()

## Save Model

In [ ]:
# Save the pipeline
model_path = MODEL_DIR / 'baseline_tfidf_logreg.joblib'
joblib.dump(pipeline, model_path)
print(f'Model saved to: {model_path}')
print(f'Model size: {model_path.stat().st_size / 1024 / 1024:.1f} MB')

In [ ]:
# Quick inference test
loaded_pipeline = joblib.load(model_path)

test_texts = [
    "Scientists confirm that climate change is accelerating faster than predicted",
    "SHOCKING: Government hiding alien technology in secret underground base!!!",
    "The president signed a new infrastructure bill into law today",
    "You won't BELIEVE what this celebrity did - doctors HATE this trick!"
]

predictions = loaded_pipeline.predict(test_texts)
probabilities = loaded_pipeline.predict_proba(test_texts)

for text, pred, prob in zip(test_texts, predictions, probabilities):
    label = 'FAKE' if pred == 1 else 'REAL'
    confidence = prob[pred] * 100
    print(f'[{label} {confidence:.1f}%] {text[:80]}...')

## Top Features

In [ ]:
# Top features for each class
feature_names = pipeline.named_steps['tfidf'].get_feature_names_out()
coefficients = pipeline.named_steps['clf'].coef_[0]

top_n = 20

# Top fake indicators (positive coefficients)
top_fake_idx = np.argsort(coefficients)[-top_n:][::-1]
top_fake = [(feature_names[i], coefficients[i]) for i in top_fake_idx]

# Top real indicators (negative coefficients)
top_real_idx = np.argsort(coefficients)[:top_n]
top_real = [(feature_names[i], coefficients[i]) for i in top_real_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 8))

# Fake indicators
words, weights = zip(*top_fake)
axes[0].barh(range(top_n), weights, color='#f44336')
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(words)
axes[0].set_title('Top 20 Fake News Indicators')
axes[0].invert_yaxis()

# Real indicators
words, weights = zip(*top_real)
axes[1].barh(range(top_n), [abs(w) for w in weights], color='#4caf50')
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(words)
axes[1].set_title('Top 20 Real News Indicators')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()